# Optimal Color Pooling via Greedy Divergence Minimization

Learn an optimal color clustering that reduces:
- Statewide JSD (county vs baseline)
- Neighbor JSD (adjacent county pairs)
- Conditional divergence (county vs neighbor-pooled)

Uses a greedy agglomerative merge algorithm with candidate restriction (Jaccard similarity).

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial.distance import jensenshannon
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'methods':
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
elif PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'results').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent  # fallback
FIGURES_DIR = PROJECT_ROOT / 'figures' / 'method_comparison'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load counts: county, landcover, color, count
counts_path = PROJECT_ROOT / 'results' / 'tables' / 'bayesian_shrinkage' / 'bayesian_shrinkage_aggregated_counts.csv'
if not counts_path.exists():
    counts_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'bayesian_shrinkage_aggregated_counts.csv'

df_counts = pd.read_csv(counts_path)
df_counts = df_counts.rename(columns={'fips': 'county', 'lc_type': 'landcover', 'clr': 'color'})
df_counts['county'] = df_counts['county'].astype(str).str.zfill(5)

# Load neighbors
neighbors_path = PROJECT_ROOT / 'website' / 'backend' / 'data' / 'ca_county_neighbors.csv'
neighbors_df = pd.read_csv(neighbors_path)
neighbors_df['county_fips'] = neighbors_df['county_fips'].astype(str).str.zfill(5)
neighbors_df['neighbor_fips'] = neighbors_df['neighbor_fips'].astype(str).str.zfill(5)

neighbors = {}
for _, r in neighbors_df.iterrows():
    c = r['county_fips']
    n = r['neighbor_fips']
    neighbors.setdefault(c, set()).add(n)
    neighbors.setdefault(n, set()).add(c)
neighbors = {k: list(v) for k, v in neighbors.items()}

## 2. Core Functions: Divergence and Smoothing

In [ ]:
def js_divergence(p, q, eps=1e-12):
    """Jensen-Shannon divergence in nats. p, q are probability vectors."""
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum()
    q = q / q.sum()
    return float(jensenshannon(p, q, base=np.e))

def laplace_smooth(counts, alpha, K=None):
    """Smooth counts to probabilities: p_k = (y_k + alpha) / (n + alpha*K)."""
    counts = np.asarray(counts, dtype=float)
    if K is None:
        K = len(counts)
    n = counts.sum()
    return (counts + alpha) / (n + alpha * K)

## 3. Build Arrays and Index Maps

In [ ]:
counties = sorted(df_counts['county'].unique())
landcovers = sorted(df_counts['landcover'].unique())
colors = sorted(df_counts['color'].unique())

county2i = {c: i for i, c in enumerate(counties)}
lc2j = {l: j for j, l in enumerate(landcovers)}
color2k = {k: idx for idx, k in enumerate(colors)}
k2color = {idx: k for k, idx in color2k.items()}

nC, nL, nK = len(counties), len(landcovers), len(colors)
Y = np.zeros((nC, nL, nK))
for _, r in df_counts.iterrows():
    i = county2i[r['county']]
    j = lc2j[r['landcover']]
    k = color2k[r['color']]
    Y[i, j, k] = r['count']

N = Y.sum(axis=2)  # N[c,l] = exposure

## 4. Compute Objectives (D_state, D_neighbor, D_cond)

In [ ]:
def compute_objectives(Yg, neighbors, county2i, lc2j, alpha, min_n, w_state=1.0, w_neighbor=1.0, w_cond=1.0):
    """
    Yg: (nC, nL, nG) grouped counts. nG = number of current color groups.
    Returns D_state, D_neighbor, D_cond, and combined score.
    """
    nC, nL, nG = Yg.shape
    
    # Statewide baseline by landcover
    Y_state = Yg.sum(axis=0)  # (nL, nG)
    N_state = Y_state.sum(axis=1)
    p0 = np.zeros((nL, nG))
    for j in range(nL):
        p0[j] = laplace_smooth(Y_state[j], alpha, nG)
    
    D_state_vals = []
    D_state_weights = []
    for c in range(nC):
        for l in range(nL):
            if N[c, l] < min_n:
                continue
            p_cl = laplace_smooth(Yg[c, l, :], alpha, nG)
            js = js_divergence(p_cl, p0[l])
            D_state_vals.append(js)
            D_state_weights.append(N[c, l])
    D_state = np.average(D_state_vals, weights=D_state_weights) if D_state_vals else 0.0
    
    # Neighbor divergence: unique pairs (c, c')
    edges = set()
    for c_str, nbrs in neighbors.items():
        if c_str not in county2i:
            continue
        for n_str in nbrs:
            if n_str not in county2i:
                continue
            pair = tuple(sorted([c_str, n_str]))
            edges.add(pair)
    
    D_neighbor_vals = []
    for (ca, cb) in edges:
        i, j = county2i[ca], county2i[cb]
        for l in range(nL):
            if N[i, l] < min_n or N[j, l] < min_n:
                continue
            p_a = laplace_smooth(Yg[i, l, :], alpha, nG)
            p_b = laplace_smooth(Yg[j, l, :], alpha, nG)
            D_neighbor_vals.append(js_divergence(p_a, p_b))
    D_neighbor = np.mean(D_neighbor_vals) if D_neighbor_vals else 0.0
    
    # Conditional: county vs neighbor-pooled
    D_cond_vals = []
    D_cond_weights = []
    for c_str, nbrs in neighbors.items():
        if c_str not in county2i:
            continue
        i = county2i[c_str]
        nbr_indices = [county2i[n] for n in nbrs if n in county2i]
        for l in range(nL):
            if N[i, l] < min_n:
                continue
            Y_pool = Yg[i, l, :].copy()
            for j in nbr_indices:
                Y_pool += Yg[j, l, :]
            p_cl = laplace_smooth(Yg[i, l, :], alpha, nG)
            p_pool = laplace_smooth(Y_pool, alpha, nG)
            D_cond_vals.append(js_divergence(p_cl, p_pool))
            D_cond_weights.append(N[i, l])
    D_cond = np.average(D_cond_vals, weights=D_cond_weights) if D_cond_vals else 0.0
    
    score = w_state * D_state + w_neighbor * D_neighbor + w_cond * D_cond
    return D_state, D_neighbor, D_cond, score

## 5. Build Candidate Merge Pairs (Jaccard)

In [ ]:
def build_candidate_pairs(Y, color2k, k2color, thresh=0.6):
    """Pairs of colors with Jaccard(county presence) > thresh."""
    nC, nL, nK = Y.shape
    # County presence: county has color k if any count > 0 for that county
    presence = (Y.sum(axis=1) > 0)  # (nC, nK)
    
    candidates = []
    for k1 in range(nK):
        for k2 in range(k1 + 1, nK):
            a = set(np.where(presence[:, k1])[0])
            b = set(np.where(presence[:, k2])[0])
            if not a or not b:
                continue
            jaccard = len(a & b) / len(a | b)
            if jaccard >= thresh:
                candidates.append((k1, k2))
    return candidates

## 6. Greedy Color Pooling

In [ ]:
def greedy_color_pooling(Y, neighbors, county2i, lc2j, k2color, alpha=1.0, min_n=30, w=(1,1,1),
                         K_target=None, max_merges=None, candidate_thresh=0.6):
    """
    Returns: color_to_group (dict str->int), group_names, merge_log (DataFrame), Yg_final
    """
    w_state, w_neighbor, w_cond = w
    nC, nL, nK = Y.shape
    
    # Initial: each color is its own group
    groups = [[k] for k in range(nK)]  # list of lists of original color indices
    G = len(groups)
    Yg = Y.copy()  # (nC, nL, nK) -> will collapse to (nC, nL, G)
    
    def get_group_name(g):
        return '+'.join(sorted(k2color[k] for k in g))
    
    candidates = build_candidate_pairs(Y, None, k2color, candidate_thresh)
    # Map to group indices: initially group g contains color g
    color_to_group_idx = {k: k for k in range(nK)}
    
    log_rows = []
    step = 0
    Ds, Dn, Dc, score = compute_objectives(Yg, neighbors, county2i, lc2j, alpha, min_n, w_state, w_neighbor, w_cond)
    
    while True:
        step += 1
        if K_target and G <= K_target:
            break
        if max_merges and step > max_merges:
            break
        
        # Candidate pairs: only between current groups (by original color indices)
        # For groups, we need pairs of groups that we can merge. Use group indices.
        # Build candidate group pairs from candidate color pairs
        cand_group_pairs = set()
        for k1, k2 in candidates:
            g1 = color_to_group_idx.get(k1)
            g2 = color_to_group_idx.get(k2)
            if g1 is not None and g2 is not None and g1 != g2:
                cand_group_pairs.add(tuple(sorted([g1, g2])))
        
        if not cand_group_pairs:
            break
        
        best_delta = -np.inf
        best_pair = None
        best_Yg_new = None
        best_Ds, best_Dn, best_Dc = None, None, None
        
        for (ga, gb) in cand_group_pairs:
            if ga >= G or gb >= G:
                continue
            # Simulate merge: combine group ga and gb
            Yg_new = np.delete(Yg, gb, axis=2)
            Yg_new[:, :, ga] += Yg[:, :, gb]
            G_new = Yg_new.shape[2]
            Ds_new, Dn_new, Dc_new, score_new = compute_objectives(
                Yg_new, neighbors, county2i, lc2j, alpha, min_n, w_state, w_neighbor, w_cond)
            delta = score - score_new
            if delta > best_delta:
                best_delta = delta
                best_pair = (ga, gb)
                best_Yg_new = Yg_new
                best_Ds, best_Dn, best_Dc = Ds_new, Dn_new, Dc_new
        
        if best_delta <= 0 or best_pair is None:
            break
        
        ga, gb = best_pair
        name_a = get_group_name(groups[ga])
        name_b = get_group_name(groups[gb])
        
        log_rows.append({
            'merge_step': step, 'group_a': name_a, 'group_b': name_b,
            'score_before': score, 'score_after': score - best_delta, 'delta': best_delta,
            'D_state_before': Ds, 'D_state_after': best_Ds,
            'D_neighbor_before': Dn, 'D_neighbor_after': best_Dn,
            'D_cond_before': Dc, 'D_cond_after': best_Dc
        })
        
        # Apply merge
        groups[ga] = groups[ga] + groups[gb]
        del groups[gb]
        Yg = best_Yg_new
        G = Yg.shape[2]
        Ds, Dn, Dc = best_Ds, best_Dn, best_Dc
        score = score - best_delta
        
        # Update color_to_group_idx: colors in deleted group -> ga; indices > gb shift down
        for k in range(nK):
            if color_to_group_idx[k] == gb:
                color_to_group_idx[k] = ga
            elif color_to_group_idx[k] > gb:
                color_to_group_idx[k] -= 1
    
    # Final mapping: color name -> group index (or group name)
    color_to_group = {}
    for k in range(nK):
        gidx = color_to_group_idx[k]
        # Map to final group index (groups may have been reordered)
        color_to_group[k2color[k]] = gidx
    
    group_names = [get_group_name(g) for g in groups]
    merge_log = pd.DataFrame(log_rows)
    return color_to_group, group_names, merge_log, Yg, groups

## 7. Run Optimization

In [ ]:
alpha = 1.0
min_n = 30
w = (1.0, 1.0, 1.0)
candidate_thresh = 0.5

color_to_group, group_names, merge_log, Yg_final, groups_final = greedy_color_pooling(
    Y, neighbors, county2i, lc2j, k2color, alpha=alpha, min_n=min_n, w=w,
    K_target=None, max_merges=25, candidate_thresh=candidate_thresh)

## 8. Diagnostics and Results

In [ ]:
merge_log

In [ ]:
Ds_before, Dn_before, Dc_before, score_before = compute_objectives(
    Y, neighbors, county2i, lc2j, alpha, min_n, *w)
Ds_after, Dn_after, Dc_after, score_after = compute_objectives(
    Yg_final, neighbors, county2i, lc2j, alpha, min_n, *w)

results_summary = {
    'Before': {'D_state': Ds_before, 'D_neighbor': Dn_before, 'D_cond': Dc_before, 'Score': score_before},
    'After': {'D_state': Ds_after, 'D_neighbor': Dn_after, 'D_cond': Dc_after, 'Score': score_after}
}
pd.DataFrame(results_summary)

## 9. Visualizations

### 9.1 Before vs After Neighbor Divergence Histogram

In [ ]:
def get_neighbor_jsd_values(Yg, neighbors, county2i, N, alpha, min_n):
    """Get per-pair JSD values for histogram."""
    nC, nL, nG = Yg.shape
    edges = set()
    for c_str, nbrs in neighbors.items():
        if c_str not in county2i:
            continue
        for n_str in nbrs:
            if n_str not in county2i:
                continue
            edges.add(tuple(sorted([c_str, n_str])))
    vals = []
    for (ca, cb) in edges:
        i, j = county2i[ca], county2i[cb]
        for l in range(nL):
            if N[i, l] < min_n or N[j, l] < min_n:
                continue
            p_a = laplace_smooth(Yg[i, l, :], alpha, nG)
            p_b = laplace_smooth(Yg[j, l, :], alpha, nG)
            vals.append(js_divergence(p_a, p_b))
    return vals

jsd_before = get_neighbor_jsd_values(Y, neighbors, county2i, N, alpha, min_n)
jsd_after = get_neighbor_jsd_values(Yg_final, neighbors, county2i, N, alpha, min_n)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(jsd_before, bins=35, alpha=0.6, color='#E74C3C', label=f'Before (mean={np.mean(jsd_before):.3f})', edgecolor='white')
ax.hist(jsd_after, bins=35, alpha=0.6, color='#3498DB', label=f'After (mean={np.mean(jsd_after):.3f})', edgecolor='white')
ax.set_xlabel('Neighbor pair JSD (nats)')
ax.set_ylabel('Count')
ax.set_title('Before vs After Color Pooling: Neighbor JSD Distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pooling_before_after_neighbor_jsd.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.2 Score vs Number of Groups Curve

In [ ]:
if len(merge_log) > 0:
    log = merge_log.copy()
    log['n_groups'] = nK - log['merge_step']
    n_grp = [nK] + log['n_groups'].tolist()
    scores = [score_before] + log['score_after'].tolist()
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(n_grp, scores, 'o-', color='#2E86AB', linewidth=2, markersize=6)
    ax.set_xlabel('Number of color groups')
    ax.set_ylabel('Combined score')
    ax.set_title('Score vs Number of Groups (Greedy Pooling)')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'pooling_score_vs_groups.png', dpi=150, bbox_inches='tight')
    plt.show()

### 9.3 Color Group Heatmap (Group × Landcover Proportions)

In [ ]:
Y_state_g = Yg_final.sum(axis=0)  # (nL, nG)
props = Y_state_g / Y_state_g.sum(axis=1, keepdims=True)
df_heat = pd.DataFrame(props, index=landcovers, columns=group_names)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(df_heat.T, ax=ax, cmap='YlOrRd', cbar_kws={'label': 'Proportion'})
ax.set_title('Pooled Color Groups: Proportion by Landcover (Statewide)')
ax.set_xlabel('Landcover')
ax.set_ylabel('Color group')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pooling_group_landcover_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 9.4 Hierarchical View of Merged Groups (Dendrogram-style)

In [ ]:
# Build linkage-like structure from merge log for visualization
if len(groups_final) > 0:
    group_sizes = [len(g) for g in groups_final]
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(range(len(group_names)), group_sizes, color='#3498DB', alpha=0.8)
    ax.set_yticks(range(len(group_names)))
    ax.set_yticklabels(group_names, fontsize=9)
    ax.set_xlabel('Number of original colors in group')
    ax.set_title('Final Color Groups (Greedy Pooling)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'pooling_final_groups.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# (Results section printed in Section 10 below)

### 9.5 County Divergence vs Exposure Scatter (Before vs After)

In [ ]:
def get_county_cond_divergence(Yg, neighbors, county2i, N, alpha, min_n):
    """Per county mean conditional divergence."""
    nC, nL, nG = Yg.shape
    county_div = []
    county_exp = []
    for c_str, nbrs in neighbors.items():
        if c_str not in county2i:
            continue
        i = county2i[c_str]
        nbr_indices = [county2i[n] for n in nbrs if n in county2i]
        divs = []
        exps = []
        for l in range(nL):
            if N[i, l] < min_n:
                continue
            Y_pool = Yg[i, l, :].copy()
            for j in nbr_indices:
                Y_pool += Yg[j, l, :]
            p_cl = laplace_smooth(Yg[i, l, :], alpha, nG)
            p_pool = laplace_smooth(Y_pool, alpha, nG)
            divs.append(js_divergence(p_cl, p_pool))
            exps.append(N[i, l])
        if divs:
            county_div.append(np.mean(divs))
            county_exp.append(np.sum(exps))
    return np.array(county_exp), np.array(county_div)

exp_before, div_before = get_county_cond_divergence(Y, neighbors, county2i, N, alpha, min_n)
exp_after, div_after = get_county_cond_divergence(Yg_final, neighbors, county2i, N, alpha, min_n)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(exp_before, div_before, alpha=0.5, s=20, c='#E74C3C', label='Before')
axes[0].set_xscale('log')
axes[0].set_xlabel('County exposure')
axes[0].set_ylabel('Mean conditional divergence')
axes[0].set_title('Before Pooling')
axes[0].grid(alpha=0.3)
axes[1].scatter(exp_after, div_after, alpha=0.5, s=20, c='#3498DB', label='After')
axes[1].set_xscale('log')
axes[1].set_xlabel('County exposure')
axes[1].set_ylabel('Mean conditional divergence')
axes[1].set_title('After Pooling')
axes[1].grid(alpha=0.3)
fig.suptitle('County Divergence vs Exposure: Before vs After Color Pooling', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pooling_divergence_vs_exposure.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Export Results for results.md

In [ ]:
# Results section for results.md
section_text = f'''
## Optimal Color Pooling (Greedy Divergence Minimization)

| Metric | Before | After |
|--------|--------|-------|
| Score | {score_before:.4f} | {score_after:.4f} |
| D_neighbor | {Dn_before:.4f} | {Dn_after:.4f} |
| D_cond | {Dc_before:.4f} | {Dc_after:.4f} |

- Initial colors: {nK} | Final groups: {len(groups_final)} | Merges: {len(merge_log)}
- Figures: `figures/method_comparison/pooling_*.png`
'''
print(section_text)